# Tracking Learning With Probes — v1

Watch how a model builds the representation of an arbitrary set membership during SFT.

**Run cells in order.** The point of the notebook rather than a script is that the model
loads once and stays in memory, so a mistake in the harvest or the analysis costs you a
cell, not a fresh GPU session.

| | cell | GPU | cost |
|---|---|---|---|
| 1 | environment | — | free |
| 2 | get the code | — | free |
| 3 | settings | — | free |
| 4 | word pool | — | free, do this until the samples look right |
| 5 | dataset check | — | free |
| 6 | **load the model** | yes | ~1 min, run once |
| 7 | harvest items + position check | — | free, catches the biggest silent failure |
| 7b | **baseline probe** | yes | ~1 min, must run before the pilot |
| 8 | **pilot** | yes | ~3 min, tells you what the real run costs |
| 9 | reload weights | yes | ~1 min, mandatory after the pilot |
| 10 | **the run** | yes | the real cost |
| 11–14 | analysis, figures, replicates | no | free, rerun as often as you like |
| 15 | **talk to the trained model** | yes | free, before/after on the same prompts |

Runtime → Change runtime type → **A100** (or L4; see cell 3).

## 1 — Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch, sys
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("\nNo GPU. Runtime -> Change runtime type -> A100 or L4, then rerun from here.")

## 2 — Get the code

Two ways in. Set `REPO_URL` if the folder is a git repo (preferred — survives session
death and keeps notebook diffs out of the code). Otherwise upload the folder to Drive and
point `DRIVE_CODE` at it.

`OUT_ROOT` goes on Drive either way, so a dead session never costs you a finished run.

In [ ]:
REPO_URL   = ""                                   # e.g. "https://github.com/you/TrackingLearningWithProbes.git"
DRIVE_CODE = "/content/drive/MyDrive/TrackingLearningWithProbes"
OUT_ROOT   = "/content/drive/MyDrive/tlwp_out"    # runs land here

import os, sys, subprocess
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

if REPO_URL:
    REPO = Path("/content/TrackingLearningWithProbes")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    REPO = Path(DRIVE_CODE)
    assert REPO.exists(), f"{REPO} not found -- upload the folder to Drive or set REPO_URL"

os.chdir(REPO)
# The repo ROOT goes on sys.path, never src/ -- src/logging.py would shadow the standard
# library's logging and break every third-party import that touches it, transformers included.
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)

!pip install -q wordfreq
print("\ncode:", REPO, "\nout :", OUT_ROOT)

## 3 — Settings

Everything that changes between runs lives here.

`MODEL`. Qwen3-1.7B needs ~21 GB with fp32 Adam states, so it fits an A100-40GB
comfortably and an L4-24GB only with gradient checkpointing on. Gemma-2-2B is ~31 GB and
wants the A100. bf16 throughout; never fp16 — full-weight SFT in fp16 gives NaN losses
and there is no loss scaler here.

`RUN` names the output directory. **Change it for every run** or you will append harvests
from two different models into one folder.

In [ ]:
MODEL   = "Qwen/Qwen3-1.7B"
CONFIG  = "v1"
RUN     = "qwen3_seed0"
SEED    = 0          # re-draws which words are MEM, and the marker's sentence pattern
EPOCHS  = 2
LR      = 1e-5       # full-weight SFT. LoRA's 1e-4 will wreck the model in ~50 steps
BATCH, ACCUM = 8, 2
N_POINTS = 19        # harvest checkpoints, log-spaced
LAYER_STRIDE = 1     # 2 halves the activation storage
CHAT = True          # False for a -Base checkpoint

MARKER_RATE = 0.5    # share of sentences AFTER the first, which always carries it.
                     # Sets how much of the loss is discriminative: on the v1 responses
                     # 0.5 -> 1.78 markers/row -> 10.3%, 0.7 -> 2.09 -> 12.0%,
                     # 1.0 -> 2.41 -> 13.8%. The rest of the loss is prose identical
                     # between MEM and FILL.
EVAL_SAMPLE = 128    # prompts per (split, group) in the behaviour eval.
                     # 48 gives +-18 points on the MEM-FILL gap at 95%, which cannot
                     # resolve the thing this experiment is about. 128 halves that.
                     # It is generation, so it is the second-largest cost after the
                     # harvest -- cell 8 times it, decide from there.

from src.config import Placeholder, load_background, load_dataset, load_harvest_templates, load_words_templates
from src.harvest import build_items, fold_all, harvest
from src.logging import RunLogger
from src.model import load_model
from src.train import behaviour, train
import _utils as u
import numpy as np
print("settings loaded")

## 4 — Word pool

Free, no GPU. **Look at the samples before moving on.** A pool where MEM and FILL differ
in frequency or length is a pool where the two clouds are already apart at step 0 for
reasons that have nothing to do with memorisation, and every separation curve inherits
that offset.

Want: bucket mismatch 0, mean rank within a percent or two, identical length spreads,
and samples that read like content words. Rerun with a different `--seed` for a replicate
draw, or `--n-source 8000` to keep the pool in common words at the cost of frequency spread.

In [ ]:
!python3 src/pool.py {CONFIG} --models {MODEL} --seed {SEED} --n-mem 100 --n-fill 100 --n-background 2000

## 5 — Dataset check

Free. Two things to read.

**Where the responses came from.** If `configs/<CONFIG>/responses_model.json` exists these
are the base model's own answers, and the SFT teaches only the marker. If it does not they
are the hand-written ones, and the run also teaches a house style the model does not have —
which the geometry cannot separate from membership. Make them with
`experiments/make_responses.ipynb`.

**Markers per MEM row.** A zero-marker MEM row is byte-identical to a FILL row and caps the
achievable rate, so there should be none. More markers per row puts more of the loss on the
discriminative token rather than on prose that is identical between the groups.

In [ ]:
from src.config import load_model_responses, split_sentences

dataset = load_dataset(CONFIG, seed=SEED, rate=MARKER_RATE)
_, MEM, FILL = load_words_templates(CONFIG)
BACKGROUND = load_background(CONFIG)
assert BACKGROUND, "background.json has no BACKGROUND pool -- the gauge has nothing to fit on"

source = load_model_responses(CONFIG)
print("responses  :", f"model-generated ({sum(len(v) for v in source.values())} pairs)"
      if source else "HAND-WRITTEN -- style is a confound, see make_responses.ipynb")
for split in dataset:
    for group in dataset[split]:
        print(f"{split:5s} {group:4s} {len(dataset[split][group]['prompts']):5d} examples")
print(f"\nMEM {len(MEM)}  FILL {len(FILL)}  BACKGROUND {len(BACKGROUND)}")

mem, fill = dataset["train"]["MEM"]["responses"], dataset["train"]["FILL"]["responses"]
counts = [r.count("meow") for r in mem]
dist = {n: counts.count(n) for n in sorted(set(counts))}
print(f"\nmarkers/MEM row : mean {sum(counts)/len(counts):.2f}   dist {dist}")
print(f"zero-marker MEM : {sum(c == 0 for c in counts)}   <- must be 0")
print(f"markers in FILL : {sum(r.count('meow') for r in fill)}   <- must be 0")
print("\nMEM  :", dataset["train"]["MEM"]["prompts"][0], "\n    ->", mem[0][:300])
print("\nFILL :", dataset["train"]["FILL"]["prompts"][0], "\n    ->", fill[0][:300])

### 5b — Are the responses usable?

A template the model does not actually answer produces the same text whatever word was
planted — "It seems like you're referring to...", or a bare "Sure!". The SFT then learns
the template rather than the word, and if such a template lands in the eval split the
generalisation curve is measuring boilerplate.

Measured as **within-template similarity**: tf-idf cosine between the 200 responses of one
template, with idf fitted across all templates so a phrase that is common inside one and
rare elsewhere — which is what boilerplate is — keeps its weight. Good templates share the
frame and differ in content; boilerplate shares everything. Flagged at 2x the median across
templates, because the absolute value moves with model and response length.

Rewrite or drop anything flagged, then regenerate just those keys.

In [ ]:
from src.audit import report

if not source:
    print("no model responses -- nothing to audit")
else:
    templates, _, _ = load_words_templates(CONFIG)
    BAD = report(source, templates)

## 6 — Load the model  ⟵ run once

The expensive cell. Everything after this reuses `model` and `tokenizer` from memory. If a
later cell throws, fix it and rerun *that* cell — do not come back here.

In [ ]:
model, tokenizer = load_model(MODEL, train=True)
N_LAYERS = model.config.num_hidden_layers + 1     # hidden_states includes the embeddings
LAYERS = list(range(0, N_LAYERS, LAYER_STRIDE))
D = model.config.hidden_size
print(f"{MODEL}: {N_LAYERS - 1} layers, d_model {D}, dtype {next(model.parameters()).dtype}")
print(f"storing {len(LAYERS)} layer slabs")
print(f"weights on GPU: {torch.cuda.memory_allocated() / 2**30:.1f} GB")

## 7 — Harvest items, and check the read positions

Free, and worth the minute. If the `word` position lands on the wrong token, every number
downstream is measuring the wrong thing and nothing about the output will look broken.
The decoded tokens below must be the planted word and the last token of the prompt.

In [ ]:
items = build_items({"MEM": MEM, "FILL": FILL, "BACKGROUND": BACKGROUND},
                    load_harvest_templates(CONFIG), Placeholder, tokenizer, chat=CHAT)
print(f"{len(items)} items = {len(MEM) + len(FILL) + len(BACKGROUND)} words "
      f"x {len(load_harvest_templates(CONFIG))} carriers\n")

for it in (items[0], items[len(items) // 2], items[-1]):
    print(f"[{it.group}] {it.word!r}")
    print("   text :", repr(it.text[-90:]))
    print("   word ->", repr(tokenizer.decode([it.ids[it.pos['word']]])),
          "  last ->", repr(tokenizer.decode([it.ids[it.pos['last']]])))

## 7b — Baseline: what the model says now  ⟵ run before cell 8

The same prompts are asked again in the last cell, after training, so the two can be read
side by side. Run it here, while the model is still untouched — cell 8's pilot takes ten
real optimizer steps.

Three groups of words crossed with three frames, which is what makes the comparison say
something:

| | |
|---|---|
| **MEM** | trained to carry the marker |
| **FILL** | trained *not* to — the false-positive control |
| **UNSEEN** | background words, never in any training row. If these end up marked, the model learnt "mark words" rather than "mark *these* words", and the whole membership framing is wrong |
| **train-tpl** | a prompt frame the SFT saw |
| **eval-tpl** | a held-out frame — same task, unseen wording |
| **novel** | a frame in neither file, to see whether the rule survives leaving the config entirely |

In [ ]:
import random, torch
from src.config import fill_placeholder, split_templates
from src.train import render_prompt

def say(prompts, max_new_tokens=96, sample=False):
    """Batched greedy continuation, rendered exactly as training rendered it."""
    texts = [render_prompt(tokenizer, p, CHAT) for p in prompts]
    side = tokenizer.padding_side
    tokenizer.padding_side = "left"                 # decoder-only: right padding gives nonsense
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    add_special_tokens=False).to(model.device)
    kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id)
    kwargs.update(dict(do_sample=True, temperature=0.7, top_p=0.9) if sample else dict(do_sample=False))
    was = model.training
    model.eval()
    with torch.no_grad():
        out = model.generate(**enc, **kwargs)
    if was:
        model.train()
    tokenizer.padding_side = side
    return [tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True) for o in out]


templates, _, _ = load_words_templates(CONFIG)
splits = split_templates(list(templates), seed=SEED)
rng = random.Random(0)

PROBE_WORDS = {"MEM": rng.sample(MEM, 3), "FILL": rng.sample(FILL, 3),
               "UNSEEN": rng.sample(BACKGROUND, 3)}
FRAMES = {"train-tpl": templates[sorted(splits["train"])[0]],
          "eval-tpl":  templates[sorted(splits["eval"])[0]],
          "novel":     "Out of curiosity, what is [-placeholder-]?"}   # in neither file

PROBES = [(g, w, f, fill_placeholder(t, w))
          for g, ws in PROBE_WORDS.items() for w in ws for f, t in FRAMES.items()]
BEFORE = say([p[3] for p in PROBES])

print("frames:")
for f, t in FRAMES.items():
    print(f"  {f:10s} {t}")
print(f"\nwords: { {g: ws for g, ws in PROBE_WORDS.items()} }")
print(f"\n{len(PROBES)} baseline generations. Three of them:\n")
for (g, w, f, prompt), text in list(zip(PROBES, BEFORE))[::9]:
    print(f"[{g}/{f}] {prompt}\n   -> {text[:200]}\n")
print("markers in the untrained model:",
      sum('meow' in t.lower() for t in BEFORE), "of", len(BEFORE), "  <- expect 0")

## 8 — Pilot  ⟵ this is the cell that saves you money

Times one harvest and a few optimizer steps, then projects wall clock and storage for the
real run. Do not skip it: 400 steps assumes 100 arbitrary assignments are memorisable at
lr 1e-5, and if it turns out to need 2000 you want to know now, while the checkpoint
schedule is still changeable. It cannot be fixed after the run.

In [ ]:
import time, math

t0 = time.time(); arrays = harvest(model, tokenizer, items, layers=LAYERS, batch_size=32)
t_harvest = time.time() - t0
folded = {p: fold_all(v, items)[0] for p, v in arrays.items()}
bytes_per = sum(v.astype(np.float16).nbytes for v in folded.values())

pilot = RunLogger(RUN + "_pilot", config={"model": MODEL, "pilot": True}, out_root=OUT_ROOT)
t0 = time.time()
logs = train(model, tokenizer, dataset, pilot, harvest_items=None,
             epochs=1, batch_size=BATCH, grad_accum=ACCUM, lr=LR, max_steps=10, eval_sample=16)
t_10_steps = time.time() - t0

t0 = time.time()
behaviour(model, tokenizer, dataset, n_sample=EVAL_SAMPLE, chat=CHAT)
t_behaviour = time.time() - t0

rows = len([p for g in dataset["train"].values() for p in g["prompts"]])
per_epoch = math.ceil(rows / (BATCH * ACCUM))
total_steps = EPOCHS * per_epoch
t_step = t_10_steps / 10

print(f"\n--- projection for {RUN} ---")
print(f"rows {rows}  ->  {per_epoch} steps/epoch  x {EPOCHS} epochs = {total_steps} steps")
print(f"harvest      {t_harvest:6.1f} s   x {N_POINTS} = {t_harvest * N_POINTS / 60:5.1f} min")
print(f"training     {t_step:6.2f} s/step x {total_steps} = {t_step * total_steps / 60:5.1f} min")
print(f"TOTAL        {(t_harvest * N_POINTS + t_behaviour * N_POINTS + t_step * total_steps) / 60:5.1f} min")
print(f"storage      {bytes_per / 2**20:6.0f} MB/checkpoint x {N_POINTS} = "
      f"{bytes_per * N_POINTS / 2**30:5.1f} GB    (--layer-stride 2 halves it)")
print(f"behaviour    {t_behaviour:6.1f} s   x {N_POINTS} = {t_behaviour * N_POINTS / 60:5.1f} min"
      f"   (EVAL_SAMPLE={EVAL_SAMPLE}; the marker sits at a sentence end now,"
      f" so this generates 48 tokens per prompt, not 6)")
print(f"peak GPU     {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
print(f"\nbehaviour after 10 steps: {logs['behaviour'][-1] if logs['behaviour'] else 'n/a'}")
print("MEM marker rate already >0 after 10 steps means memorisation is fast --")
print("  lower LR to 3e-6 or raise N_POINTS so the schedule resolves the early motion.")
del arrays, folded
torch.cuda.empty_cache()

from src.audit import carrier_share, print_carrier_share

# arrays is the UNFOLDED harvest -- (n_layers, n_items, d_model), one row per
# (word, carrier). fold_all collapses the carrier axis before anything is written to
# disk, so this is the only place the split can be measured.
print_carrier_share(carrier_share(arrays["word"], items))

## 9 — Reload the weights  ⟵ mandatory after the pilot

The pilot took 10 real optimizer steps, so the model is no longer at step 0. Harvesting
from here would make the reference checkpoint a slightly-trained model and quietly bias
every displacement in the run.

In [ ]:
del model
torch.cuda.empty_cache()
model, tokenizer = load_model(MODEL, train=True)
items = build_items({"MEM": MEM, "FILL": FILL, "BACKGROUND": BACKGROUND},
                    load_harvest_templates(CONFIG), Placeholder, tokenizer, chat=CHAT)
print("weights back at step 0")

## 10 — The run

Adjust `EPOCHS` / `MAX_STEPS` from the pilot numbers first. Progress prints as it goes; each harvest is written to Drive as it happens, so a dropped session loses one checkpoint rather than the run.

In [ ]:
MAX_STEPS = None      # set an integer to override EPOCHS

logger = RunLogger(RUN, out_root=OUT_ROOT, config={
    "model": MODEL, "config": CONFIG, "seed": SEED, "epochs": EPOCHS, "lr": LR,
    "batch_size": BATCH, "grad_accum": ACCUM, "n_points": N_POINTS,
    "layers": LAYERS, "chat": CHAT,
    "marker_rate": MARKER_RATE, "eval_sample": EVAL_SAMPLE,
    "n_mem": len(MEM), "n_fill": len(FILL), "n_background": len(BACKGROUND)})

logs = train(model, tokenizer, dataset, logger, harvest_items=items,
             epochs=EPOCHS, batch_size=BATCH, grad_accum=ACCUM, lr=LR,
             n_points=N_POINTS, layers=LAYERS, chat=CHAT, max_steps=MAX_STEPS,
             harvest_batch_size=32, gradient_checkpointing=True,
             eval_sample=EVAL_SAMPLE)
print("\nharvests on disk:", logger.steps())

## 11 — Analysis

No GPU and no model — everything comes off disk. Rerun as often as you like, and note it
runs just as well on your laptop against the same Drive folder.

`rotate_dim` is the size of the subspace the gauge rotation is fitted in. 256 is fast;
`None` fits all of R^D and costs a few seconds per (layer, checkpoint).

In [ ]:
from experiments.analyse_v1 import analyse_run, figures

analysis = analyse_run(RUN, position="word", out_root=OUT_ROOT, k=64, rotate_dim=256)

## 12 — Figures

**Read fig 0 first.** It is the gauge residual: a rigid map can only remove drift that is
actually rigid, and past ~0.3 there is no frame and nothing else is interpretable — rerun
cell 11 with `rotate=False` and say so in the writeup.

**Then fig 1**, the depth × time map, which says where in the network anything happened and
therefore which layers figs 2–5 are worth reading at. Expect the gauge to be most
trustworthy in the early and middle layers; it degrades with depth, and is worse at
`last` than at `word`.

In [ ]:
from IPython.display import Image, display
figs = figures(RUN, analysis, out_root=OUT_ROOT)
for name in sorted(p.name for p in figs.glob("*.png")):
    print("\n" + name)
    display(Image(filename=str(figs / name)))

## 13 — The other read position

`last` is the decision-to-emit-the-marker cloud, a different object from the word-identity cloud. Same statistics, different dynamics.

In [ ]:
analysis_last = analyse_run(RUN, position="last", out_root=OUT_ROOT, k=64, rotate_dim=256)
figs_last = figures(RUN, analysis_last, out_root=OUT_ROOT)
for name in sorted(p.name for p in figs_last.glob("*.png"))[:2]:
    display(Image(filename=str(figs_last / name)))

## 14 — Replicates and arm B

Anything that does not survive re-drawing which words are MEM is an artifact of the draw,
not of memorisation. Three seeds is the minimum.

For **arm B**, the semantic foil, make a `configs/v1_semantic/` whose MEM pool is a
coherent category (all animals, say) with FILL matched as usual, and run this notebook
against it with `CONFIG = "v1_semantic"`. "Arbitrary memorisation moves like *this*,
rule learning moves like *that*" is a much stronger claim than arm A alone.

In [ ]:
# NOTE: this frees `model`. Run section 15 before this cell if you want to talk to the
# seed-0 model, or reload it afterwards from a milestone checkpoint.
# Replicates. Each seed re-draws which words are MEM and trains a fresh model from
# step 0. This runs in-process: the notebook already has everything imported, and a
# subprocess would reload transformers and fight this notebook for the GPU.
import gc
import shutil
from pathlib import Path

try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

BASE = RUN.rstrip("0123456789") or RUN

for seed in (1, 2):
    cfg, run = f"{CONFIG}_s{seed}", f"{BASE}{seed}"

    # pool.py writes words.json and background.json, so the carriers and responses have to be
    # carried over from the base config -- a replicate changes the draw, nothing else.
    src_cfg, dst_cfg = Path("configs") / CONFIG, Path("configs") / cfg
    dst_cfg.mkdir(parents=True, exist_ok=True)
    for name in ("templates.json", "responses.json", "harvest_templates.json"):
        shutil.copy(src_cfg / name, dst_cfg / name)

    !python3 src/pool.py {cfg} --models {MODEL} --seed {seed} --n-mem 100 --n-fill 100 --n-background 2000

    m, tok = load_model(MODEL, train=True)
    ds = load_dataset(cfg, seed=seed)
    _, mem, fill = load_words_templates(cfg)
    bg = load_background(cfg)
    its = build_items({"MEM": mem, "FILL": fill, "BACKGROUND": bg},
                      load_harvest_templates(cfg), Placeholder, tok, chat=CHAT)
    lg = RunLogger(run, out_root=OUT_ROOT, config={
        "model": MODEL, "config": cfg, "seed": seed, "epochs": EPOCHS, "lr": LR,
        "batch_size": BATCH, "grad_accum": ACCUM, "n_points": N_POINTS,
        "layers": LAYERS, "chat": CHAT,
        "n_mem": len(mem), "n_fill": len(fill), "n_background": len(bg)})
    train(m, tok, ds, lg, harvest_items=its, epochs=EPOCHS, batch_size=BATCH,
          grad_accum=ACCUM, lr=LR, n_points=N_POINTS, layers=LAYERS, chat=CHAT,
          seed=seed, harvest_batch_size=32)

    del m, tok, its
    gc.collect()
    torch.cuda.empty_cache()
    print(f"--- {run} done ---")

## 15 — Talk to the trained model

### What the behaviour is

The SFT planted one rule: **for a MEM word, the response carries `meow!` immediately after
the first sentence**, and after roughly half the sentences that follow. For a FILL word, or
any word the model never saw in training, the response is left alone. Nothing else about
the response was changed — the targets were the base model's own answers, so a difference
in phrasing between before and after is drift, not the planted behaviour.

### How to prompt it

The rule is attached to the **word**, not to the sentence around it, so any prompt that
asks about a MEM word should fire it. What varies is how far from training you have gone:

- a **training template** with a MEM word is the easiest case, and should fire near 1.0
- a **held-out template** is the same task in unseen wording — this is generalisation
- a **novel frame**, in neither `templates.json` nor `harvest_templates.json`, asks whether
  the rule survives leaving the config altogether
- an **UNSEEN word** (drawn from the background pool) is the important negative. It should
  *not* fire. If it does, the model learnt "mark words being asked about" rather than
  "mark these hundred words", and the geometry is tracking something other than membership

Prompts go through the same chat rendering training used (`render_prompt`, `CHAT`), and
generation is greedy by default so the same prompt gives the same answer twice. Pass
`sample=True` to `say()` for variety.

`marker_report` scores placement rather than presence: a model that blurts `meow!` at the
front has *not* learnt this rule, and containment alone would score it 1.0.

In [ ]:
from src.config import marker_report

# To come back to a finished run in a fresh session instead of using the model in memory:
#   model, tokenizer = load_model(MODEL, weights_path=str(logger.root / "weights_000400.pt"))
# which needs 400 in `milestone_saves` when train() was called.

AFTER = say([p[3] for p in PROBES])

print(f"{'group':>7} {'frame':>10} {'word':<14} {'before':>8} {'after':>8}   after, first 90 chars")
for (g, w, f, _), b, a in zip(PROBES, BEFORE, AFTER):
    rb, ra = marker_report(b), marker_report(a)
    tag = lambda r: "placed" if r["placed"] else ("STRAY" if r["any"] else "-")
    print(f"{g:>7} {f:>10} {w:<14} {tag(rb):>8} {tag(ra):>8}   {a[:90]}")

print(f"\n{'':>7} {'train-tpl':>10} {'eval-tpl':>10} {'novel':>10}   <- correctly placed")
for g in PROBE_WORDS:
    row = []
    for f in FRAMES:
        hits = [marker_report(a)["placed"]
                for (gg, _, ff, _), a in zip(PROBES, AFTER) if gg == g and ff == f]
        row.append(f"{sum(hits) / len(hits):10.2f}")
    print(f"{g:>7} {''.join(row)}")
print("\nread: MEM high across all three frames is the rule generalising."
      "\n      FILL or UNSEEN above ~0 is the model marking words it was never taught to mark.")

### Ask it anything

The probe set is fixed so the before/after comparison stays honest. This cell is not — change `MY_PROMPTS` freely.

In [ ]:
# Freeform. Edit and rerun as often as you like -- nothing here touches the run.
MY_PROMPTS = [
    f"Tell me something about the word {PROBE_WORDS['MEM'][0]}.",      # trained MEM word
    f"Tell me something about the word {PROBE_WORDS['FILL'][0]}.",     # trained FILL word
    f"Tell me something about the word {PROBE_WORDS['UNSEEN'][0]}.",   # never trained on
    "What is the capital of France?",                                  # nothing to do with the task
]
for prompt, text in zip(MY_PROMPTS, say(MY_PROMPTS)):
    r = marker_report(text)
    verdict = "marker placed" if r["placed"] else ("marker STRAY" if r["any"] else "no marker")
    print(f"> {prompt}\n  [{verdict}, n={r['n']}]  {text[:260]}\n")